# Whole-Genome Phasing and Imputation

- How can measured genotypes be phased and unmeasured genotypes estimated?
- Beagle 5.5 uses Dog10K and canine genetic maps for 116 GSP and 48 LR, separately across 38 autosomes.
- Phasing estimates chromosome copies; imputation estimates unmeasured genotypes, dosages, and genotype probabilities.
- These estimates do not establish accuracy at every variant or association with training outcomes.


### (Initialization) Import Directory-Free Helpers

In [1]:
# Locate the repository and load shared helpers for the phasing and imputation workflow.

from pathlib import Path
import importlib
import sys

study_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "scripts/notebook_support.py").exists())
sys.path.insert(0, str(study_root / "scripts"))
read_json = importlib.import_module("notebook_support").read_json
study_path = importlib.import_module("notebook_support").study_path
verify_datasets = importlib.import_module("notebook_support").verify_datasets
require_saved = importlib.import_module("notebook_review").require_saved
run_script = importlib.import_module("notebook_support").run_script


### Dogs and Reference Panel

- The analysis processes 116 GSP and 48 LR separately across 38 autosomes.
- Scheme A dogs are selected afterward, preserving identical imputed genotypes across schemes.
- Training outcomes do not enter imputation.
- Dog10K uses CanFam4 and includes four GSP and six LR reference dogs.
- Accuracy requires validation.


### Match Markers to the Reference

- Positions and REF/ALT alleles must match.
- Reference markers require two SNP alleles, unique positions, matching reference bases and complete phased diploid genotypes.
- Unmatched markers are recorded separately.


### Estimate Phase and Genotypes

- Beagle 5.5 processes each chromosome using Dog10K and a canine genetic map.
- Outputs retain phased measured sites, imputed sites, dosages and genotype probabilities.


### Completed Results
- Both breeds have completed results across 38 autosomes.
- Every input marker and nonmissing imputed marker call is preserved.
- This checks input recovery, not accuracy at every imputed site.

| Breed | Dogs Analyzed | Input Markers | Imputed Markers |
| :---: | :---: | :---: | :---: |
| GSP | 116 | 155,362 | 18,009,866 |
| LR | 48 | 158,380 | 7,586,470 |


### (Initialization) Review Imputation Settings


In [2]:
# Select the imputation output folder and read the configured chromosome set.
# Identify the chromosome set and configuration used for imputation.

imputation_folder = study_root / "data/04-imputation/autosomal-20261003"
settings = read_json(study_root / "config/imputation.json")
print("Configured autosomes:", len(settings["chromosomes"]))
print("Configuration:", "config/imputation.json")

Configured autosomes: 38
Configuration: config/imputation.json


### (Imputation) Whole-Genome Phasing and Imputation
- Saved chromosome summaries describe the completed run.
- Preparation and inference commands are included as commented examples.
- *imputation_workflow.py*

In [3]:
# Reuse completed imputation outputs, or prepare and execute the configured workflow if they are incomplete.
# Check that all 38 autosomes are recorded as complete before reviewing their results.

manifest_file = imputation_folder / "manifest.private.json"
require_saved(manifest_file)
completed = manifest_file.exists() and read_json(manifest_file)["status"] == "complete"
# if not completed:
#     run_script(study_root, "imputation_workflow.py", "--output", imputation_folder,
#                "--prepare-only", linux=True)
#     run_script(study_root, "imputation_workflow.py", "--output", imputation_folder,
#                "--execute", linux=True)
imputation = read_json(manifest_file)
assert imputation["status"] == "complete"
assert set(imputation["jobs"]) == {f"chr{i}" for i in range(1, 39)}
assert all(job["status"] == "complete" for job in imputation["jobs"].values())
print("Completed:", len(imputation["jobs"]), "autosomes in both breeds.")

Completed: 38 autosomes in both breeds.


### (Consistency Check) Check Measured-Call Recovery
- Saved checks count compatible markers and preserved imputed marker calls by breed.

In [4]:
# Summarize compatible markers and preserved measured calls across chromosomes for each breed.
# These checks assess preservation of the input measurements, not accuracy at unmeasured variants.

for breed in ["GSP", "LR"]:
    records = [job["breeds"][breed] for job in imputation["jobs"].values()]
    assert all(r["verification"]["all_input_sites_recovered"] for r in records)
    markers = sum(r["panel_compatibility"]["compatible"] for r in records)
    calls = sum(r["verification"]["observed_calls_preserved"] for r in records)
    print(f"{breed}: {markers:,} compatible measured markers; {calls:,} observed calls preserved.")

GSP: 155,362 compatible measured markers; 18,009,866 observed calls preserved.
LR: 158,380 compatible measured markers; 7,586,470 observed calls preserved.
